# Jinay: Regime-specific STAD graphs (PEMS04 + PEMS08) and the PEMS08 baseline
**Team project:** TC-STAG (time-conditioned STAD graphs on DSTAGNN) | **Owner:** Jinay

You produce the **graphs our whole method depends on**, plus one early go/no-go signal for the paper. This notebook has two modes. Change `MODE` in Cell 1 and run it twice.

| | Mode `graphs` | Mode `baseline` |
|---|---|---|
| Accelerator | **None (CPU)**, which costs no GPU quota | **GPU T4 x2** |
| What it does | Fast Wasserstein (STAD) graphs: global + 3 regimes, for PEMS04 and PEMS08, plus diagnostics | Trains DSTAGNN on PEMS08, 3 seeds (baseline for our paper) |
| Time | about 10-20 min | about 1-3 h (see Cell 7 timing line) |
| Output | `jinay_graphs_outputs.zip` (**Advik needs this**) | `jinay_pems08_baseline_outputs.zip` |

**Run `graphs` first** and publish its output as a Kaggle Dataset (instructions at the end of Cell 8). It is a short CPU job, so Advik and the group get the graphs quickly.

**Kaggle settings:** Internet **On**. Add Data: a dataset with **PEMS04.npz** and one with **PEMS08.npz + PEMS08.csv** (the STSGCN data pack has both; one dataset may contain all three).

**What a "regime graph" is:** the paper computes STAD once from all training days (Eqs. 3-5) and freezes it. Here the same pipeline is run separately on (a) weekday peak hours (7-10h and 16-19h), (b) weekday off-peak, and (c) weekends, using training days only (no leakage).

## Cell 1: Settings

In [ ]:
import os, sys, json, time, subprocess, shutil, glob, threading, queue, zipfile
from pathlib import Path
import numpy as np, pandas as pd

MODE = 'graphs'          # 'graphs' (CPU session)  or  'baseline' (GPU session)
SPARSITY = {'PEMS04': 0.01, 'PEMS08': 0.02}   # PEMS04: paper's Psp (3 nodes). PEMS08: 0.02 gives the same 3 nodes (0.01 would give int(1.7)=1, see Cell 6 note)
LARGE_SPARSITY = 0.05    # second, denser top-k used for diagnostics and saved as extra graphs
TRAIN_FRAC = 0.6         # same split as the paper: graphs use the first 60% of time steps only
PEAK_HOURS = [(7, 10), (16, 19)]
START_DOW_OVERRIDE = {'PEMS04': 0, 'PEMS08': 4}   # PEMS04 = 0 CONFIRMED from the data (Mon 2018-01-01; Mondays are low, which fooled the automatic detector). PEMS08 = 4 is the documented start (Fri 2016-07-01), pending the same check
ASSUMED_START_DOW = {'PEMS04': 0, 'PEMS08': 4}   # 0=Mon. PEMS04 starts Mon 2018-01-01; PEMS08 starts Fri 2016-07-01 (verified in Cell 4)
# baseline-mode settings (must equal Harshith's):
SEEDS = [1, 2, 3]; MAX_EPOCHS = 60; PATIENCE = 15; SESSION_HOURS_BUDGET = 10.5; RUN_TIMING_TEST = True
EXPECT = {'PEMS04': (16992, 307, 3), 'PEMS08': (17856, 170, 3)}
PAPER = {'PEMS08': dict(MAE=15.67, MAPE=9.94, RMSE=24.77)}   # Table 2, DSTAGNN

DATASETS = ['PEMS04', 'PEMS08'] if MODE == 'graphs' else ['PEMS08']
DS_REG = DATASETS if MODE == 'graphs' else []   # regime graphs / diagnostics only in graphs mode
ON_KAGGLE = os.path.exists('/kaggle')
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('./working'); WORK.mkdir(exist_ok=True, parents=True)
REPO = WORK / 'DSTAGNN'; RUNS = WORK / 'runs'; RUNS.mkdir(exist_ok=True)
OUT = WORK / 'regime_graphs'; OUT.mkdir(exist_ok=True)
SESSION_DEADLINE = time.time() + SESSION_HOURS_BUDGET * 3600
N_CPU = os.cpu_count() or 1
print('MODE =', MODE, '| datasets:', DATASETS, '| CPUs:', N_CPU)

## Cell 2: Setup (authors' repo, POT optimal-transport library, shared helper module)

In [ ]:
if not REPO.exists():
    r = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/SYLan2019/DSTAGNN.git', str(REPO)], capture_output=True, text=True)
    assert REPO.exists(), 'git clone failed: turn Internet ON in settings. ' + r.stderr
try:
    import ot
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pot'], check=True); import ot
print('POT', ot.__version__)

STAD_LIB = r'''import numpy as np, ot, warnings
from multiprocessing import Pool
warnings.filterwarnings('ignore')

def prep_node_vectors(data, eps=1e-8):
    # data: (days, L, N) -> unit vectors Xn (N, days, L) and masses P (N, days)
    X = np.ascontiguousarray(np.transpose(data, (2, 0, 1)).astype(np.float64))
    nrm = np.linalg.norm(X, axis=2)                                   # (N, days) daily volume (Eq. 3)
    Xn = X / np.maximum(nrm, eps)[:, :, None]
    s = nrm.sum(1, keepdims=True)
    P = np.where(s > 0, nrm / np.maximum(s, eps), 1.0 / X.shape[1])
    return np.ascontiguousarray(Xn), np.ascontiguousarray(P)

_G = {}
def _row(i):
    Xn, P = _G['Xn'], _G['P']; N = Xn.shape[0]; out = np.zeros(N)
    for j in range(i + 1, N):
        C = np.ascontiguousarray(1.0 - Xn[i] @ Xn[j].T)                                   # cosine cost (Eq. 4)
        out[j] = 1.0 - ot.emd2(P[i], P[j], C)                          # similarity = 1 - Wasserstein (Eq. 5)
    return i, out

def stad_matrix(data, n_proc=1):
    # returns symmetric (N,N) similarity A_STAD with zero diagonal
    Xn, P = prep_node_vectors(data); _G['Xn'], _G['P'] = Xn, P; N = Xn.shape[0]
    d = np.zeros((N, N))
    if n_proc > 1:
        with Pool(n_proc) as pool:
            for i, r in pool.imap_unordered(_row, range(N), chunksize=2): d[i] = r
    else:
        for i in range(N): d[i] = _row(i)[1]
    return np.nan_to_num(d + d.T)

def make_graphs(sta, sparsity=0.01, stag_include_self=True):
    # Returns (A_STAG, A_STRG).
    # A_STRG (attention prior): authors' released STAG_gen.py logic = top-k OTHER nodes, plus the diagonal (weighted).
    # A_STAG (Chebyshev graph): the authors' SHIPPED PEMS04 file is "self + (k-1) others" (verified: 91% of rows identical when
    #   the diagonal similarity is set to 1), so with stag_include_self=True we reproduce that convention. Self-loops cancel in L = D - A,
    #   so this means k-1 real neighbours per node in the graph convolution.
    N = sta.shape[0]; I = np.identity(N)
    adjl = sta + I
    adjlnormd = adjl / adjl.mean(axis=0)
    adj = 1 - adjl + I
    A = np.zeros((N, N)); R = np.zeros((N, N)); top = int(N * sparsity)
    s2 = sta.copy(); np.fill_diagonal(s2, 1.0)
    for i in range(N):
        a = adj[i, :].argsort()[0:top]
        R[i, a] = adjlnormd[i, a]
        R[i, i] = adjlnormd[i, i]
        if stag_include_self:
            A[i, np.argsort(-s2[i], kind='stable')[:top]] = 1
        else:
            A[i, a] = 1
    return A, R
'''
(WORK / 'stad_lib.py').write_text(STAD_LIB); sys.path.insert(0, str(WORK))
from stad_lib import stad_matrix, make_graphs
if MODE == 'baseline':
    import torch
    N_GPU = torch.cuda.device_count(); print('GPUs:', N_GPU); assert N_GPU >= 1, 'baseline mode needs Accelerator = GPU T4 x2'

## Cell 3: Find and validate the data files
Needs `<DS>.npz` (shape check) and, for PEMS08, `PEMS08.csv` (road-network edges used by the static graph / Chebyshev setup). PEMS04's csv ships with the repo.

In [ ]:
def find(name):
    pat = '/kaggle/input/**/%s' % name if ON_KAGGLE else './**/%s' % name
    h = [p for p in glob.glob(pat, recursive=True) if (ON_KAGGLE or 'DSTAGNN' not in p)]   # on Kaggle only /kaggle/input is searched, so a dataset folder named DSTAGNN is fine
    return h[0] if h else None

RAW = {}
for ds in DATASETS:
    f = find(ds + '.npz'); assert f, 'Could not find %s.npz: Add Data > search "%s" and add a dataset containing it, then rerun.' % (ds, ds)
    RAW[ds] = np.load(f)['data']
    assert RAW[ds].shape == EXPECT[ds], 'Bad shape %s for %s, expected %s' % (RAW[ds].shape, ds, EXPECT[ds])
    d = REPO / 'data' / ds; d.mkdir(parents=True, exist_ok=True); shutil.copy(f, d / (ds + '.npz'))
    if not (d / (ds + '.csv')).exists():
        c = find(ds + '.csv'); assert c, 'Missing %s.csv (road edges). It is in the STSGCN data pack; add a dataset that has it.' % ds
        shutil.copy(c, d / (ds + '.csv'))
    print(ds, 'OK', RAW[ds].shape, '| csv header:', open(d / (ds + '.csv')).readline().strip())

## Cell 4: Detect the weekday offset (so "weekend" really means Sat/Sun)
Picks the offset where the 2 weekend days have the lowest traffic. The result must agree with the assumed offset; if it does not, we use the detected one and you tell the group.

In [ ]:
START_DOW = {}
for ds in DATASETS:
    T, N, _ = RAW[ds].shape; nd = T // 288
    daily = RAW[ds][:nd * 288, :, 0].reshape(nd, 288, N).mean(axis=(1, 2))
    ratio = {}
    for o in range(7):
        dow = (np.arange(nd) + o) % 7
        ratio[o] = daily[dow >= 5].mean() / daily[dow < 5].mean()
    best = min(ratio, key=ratio.get); second = sorted(ratio.values())[1]
    START_DOW[ds] = START_DOW_OVERRIDE.get(ds, best)
    print(ds, '| weekend/weekday flow ratio by offset:', {k: round(v, 3) for k, v in ratio.items()})
    print('   detected offset', best, '| assumed', ASSUMED_START_DOW[ds], '| margin to 2nd best: %.3f' % (second - ratio[best]),
          '| ' + ('AGREES' if best == ASSUMED_START_DOW[ds] else '*** DISAGREES with the documented start date: run the alignment-check notebook ***'), '| USING', START_DOW[ds])

## Cell 5: Check that our fast STAD (POT) matches the authors' slow LP version, on the real data
The authors' `STAG_gen.py` solves one scipy `linprog` per node pair (very slow). We use `ot.emd2`, which solves the same problem. This cell compares both on 4 random pairs. **Expect differences below 1e-6.**

In [ ]:
from scipy.optimize import linprog
def authors_wd(p, q, D):
    A_eq = []
    for i in range(len(p)):
        A = np.zeros_like(D); A[i, :] = 1; A_eq.append(A.reshape(-1))
    for i in range(len(q)):
        A = np.zeros_like(D); A[:, i] = 1; A_eq.append(A.reshape(-1))
    A_eq = np.array(A_eq); b = np.concatenate([p, q])
    return linprog(D.reshape(-1), A_eq=A_eq[:-1], b_eq=b[:-1]).fun

def train_days(ds):
    T, N, _ = RAW[ds].shape; nd = int(int(T * TRAIN_FRAC) / 288)
    return RAW[ds][:nd * 288, :, 0].reshape(nd, 288, N), nd

rng = np.random.default_rng(0); worst = 0
for ds in DATASETS:
    data, nd = train_days(ds); N = data.shape[2]
    pairs = [tuple(sorted(rng.choice(N, 2, replace=False))) for _ in range(4)]
    sub = data[:, :, sorted({i for p in pairs for i in p})]; idx = {n: k for k, n in enumerate(sorted({i for p in pairs for i in p}))}
    sta = stad_matrix(sub)
    for a, b in pairs:
        x, y = data[:, :, a], data[:, :, b]
        xn = np.maximum((x ** 2).sum(1, keepdims=True) ** .5, 1e-8); yn = np.maximum((y ** 2).sum(1, keepdims=True) ** .5, 1e-8)
        ref = 1 - authors_wd(xn[:, 0] / xn.sum(), yn[:, 0] / yn.sum(), 1 - (x / xn) @ (y / yn).T)
        diff = abs(ref - sta[idx[a], idx[b]]); worst = max(worst, diff)
    print(ds, 'train days:', nd, '| pairs checked:', pairs)
print('max |ours - authors| = %.2e' % worst); assert worst < 1e-5, 'POT result differs from the authors LP: stop and tell the group'

## Cell 6: Global STAD graph (all training days) and comparison with the authors' shipped PEMS04 graph
- PEMS04: we recompute the global graph and compare it with the authors' `stag_001_PEMS04.csv` (a reproduction check).
- PEMS08: the authors do **not** ship their graph, so ours is used (written where the DSTAGNN config expects it).

**Graph conventions (verified against the authors' shipped PEMS04 files):** the Chebyshev graph A_STAG is *self + (k-1) most similar nodes* (our recomputation matches 91% of the shipped rows exactly); the attention prior A_STRG is *k most similar other nodes + the diagonal* (matches 90%). **Sparsity:** Psp = 0.01 gives k = `int(0.01*N)` = 3 for PEMS04 but only 1 for PEMS08 (N=170), which would leave PEMS08 with no real graph neighbours, so we use Psp = 0.02 (k = 3) for PEMS08. Mention this as a deviation in the paper.

In [ ]:
def write_graph_csvs(A, R, path_stag, path_strg):
    pd.DataFrame(A).to_csv(path_stag, index=False, header=None); pd.DataFrame(R).to_csv(path_strg, index=False, header=None)

STAD, GRAPHS = {}, {}
for ds in DATASETS:
    data, nd = train_days(ds); t0 = time.time()
    sta = stad_matrix(data, n_proc=N_CPU); STAD[(ds, 'global')] = sta
    A, R = make_graphs(sta, SPARSITY[ds]); GRAPHS[(ds, 'global')] = (A, R)
    write_graph_csvs(A, R, REPO / 'data' / ds / ('stag_001_%s.csv' % ds) if ds != 'PEMS04' else OUT / ('ours_stag_001_%s.csv' % ds),
                     REPO / 'data' / ds / ('strg_001_%s.csv' % ds) if ds != 'PEMS04' else OUT / ('ours_strg_001_%s.csv' % ds))
    print('%s global STAD: %.0f s | neighbours per node: %d' % (ds, time.time() - t0, A[0].sum()))

if 'PEMS04' in DATASETS:
    shipped = pd.read_csv(REPO / 'data' / 'PEMS04' / 'stag_001_PEMS04.csv', header=None).values > 0
    ours = GRAPHS[('PEMS04', 'global')][0] > 0
    exact = np.mean([(shipped[i] == ours[i]).all() for i in range(len(ours))])
    part = np.mean([(shipped[i] & ours[i]).sum() / max(shipped[i].sum(), 1) for i in range(len(ours))])
    print('\nREPRODUCTION CHECK vs authors shipped PEMS04 graph: rows identical %.1f%% | mean neighbour overlap %.1f%%' % (100 * exact, 100 * part))
    print('(Expected about 90%: the rest are exact similarity ties. 60-85%: tell the group. <50%: something is wrong.)')

## Cell 7: The regime graphs (our contribution's core)
Same pipeline, but each node's daily vector is restricted to a regime: **weekday peak** (weekdays x 7-10h/16-19h), **weekday off-peak** (weekdays x all other hours), **weekend** (Sat/Sun x all hours). Training days only.

In [ ]:
def regime_slices(ds):
    nd = train_days(ds)[1]; dow = (np.arange(nd) + START_DOW[ds]) % 7
    peak = np.concatenate([np.arange(a * 12, b * 12) for a, b in PEAK_HOURS]); off = np.setdiff1d(np.arange(288), peak)
    wk, we = np.where(dow < 5)[0], np.where(dow >= 5)[0]
    return {'weekday_peak': (wk, peak), 'weekday_offpeak': (wk, off), 'weekend': (we, np.arange(288))}

REG = ['weekday_peak', 'weekday_offpeak', 'weekend']
for ds in DS_REG:
    data, nd = train_days(ds)
    for name, (days_idx, slots) in regime_slices(ds).items():
        t0 = time.time(); sub = data[np.ix_(days_idx, slots)]               # (days, slots, N)
        sta = stad_matrix(sub, n_proc=N_CPU); STAD[(ds, name)] = sta
        for tag, sp in [('', SPARSITY[ds]), ('_k5', LARGE_SPARSITY)]:
            A, R = make_graphs(sta, sp); GRAPHS[(ds, name + tag)] = (A, R)
        print('%s %-16s days=%2d slots=%3d | %.0f s' % (ds, name, len(days_idx), len(slots), time.time() - t0))
    for tag, sp in [('', SPARSITY[ds]), ('_k5', LARGE_SPARSITY)]:
        GRAPHS[(ds, 'global' + tag)] = make_graphs(STAD[(ds, 'global')], sp)
print('\nweekend graphs use few days (about 10): treat them as noisier; mention this in the paper limitations.')

## Cell 8: Diagnostics: do the regime graphs actually differ from the global graph? (go / no-go for the paper)
Mean Jaccard overlap of each node's neighbour set. **1.0 = identical graphs.** If the regime graphs are almost the same as the global one (overlap > 0.9 at the larger k), the "time-conditioned graph" idea is weak, and **you must tell the group immediately** so we can switch to a different angle early. Then everything is saved.

In [ ]:
def nbr_sets(sta, k):
    s = sta.copy(); np.fill_diagonal(s, -1); return [set(np.argsort(-s[i])[:k]) for i in range(len(s))]
def jacc(a, b): return float(np.mean([len(x & y) / max(len(x | y), 1) for x, y in zip(a, b)]))

rows = []
for ds in DS_REG:
    N = STAD[(ds, 'global')].shape[0]
    for tag, sp in [('k=%g%%' % (100 * SPARSITY[ds]), SPARSITY[ds]), ('k=5%', LARGE_SPARSITY)]:
        k = max(int(N * sp), 1); S = {r: nbr_sets(STAD[(ds, r)], k) for r in ['global'] + REG}
        for r in REG: rows.append(dict(dataset=ds, k=tag, k_nodes=k, pair='global vs ' + r, jaccard=jacc(S['global'], S[r])))
        for i, a in enumerate(REG):
            for b in REG[i + 1:]: rows.append(dict(dataset=ds, k=tag, k_nodes=k, pair=a + ' vs ' + b, jaccard=jacc(S[a], S[b])))
diag = pd.DataFrame(rows)
if len(diag): print(diag.round(3).to_string(index=False)); diag.to_csv(OUT / 'graph_overlap_diagnostics.csv', index=False)

vs_g = diag[(diag.k == 'k=5%') & diag.pair.str.startswith('global')].jaccard.mean() if len(diag) else 1.0
if len(diag): print('\nAVERAGE regime-vs-global overlap at k=5%%: %.2f' % vs_g)
if len(diag): print('=> ' + ('Regime graphs are LARGELY DIFFERENT from the global graph: good for our idea.' if vs_g < 0.7 else
      'Moderately different. OK, but report it honestly.' if vs_g < 0.9 else 'Nearly identical to global: TELL THE GROUP, the novelty claim is weak.'))

# ---- save everything Advik needs ----
for ds in DATASETS:
    d = OUT / ds; d.mkdir(exist_ok=True)
    for (dd, name), sta in STAD.items():
        if dd == ds: np.save(d / ('stad_%s.npy' % name), sta.astype(np.float32))
    for (dd, name), (A, R) in GRAPHS.items():
        if dd == ds: write_graph_csvs(A, R, d / ('stag_001_%s_%s.csv' % (ds, name)), d / ('strg_001_%s_%s.csv' % (ds, name)))
json.dump(dict(start_dow=START_DOW, sparsity=SPARSITY, large_sparsity=LARGE_SPARSITY, peak_hours=PEAK_HOURS, train_frac=TRAIN_FRAC,
               regimes=REG, note='file names: stad_<regime>.npy = dense NxN similarity (zero diag); stag_*/strg_* csv = authors format (no header). *_k5 = 5% top-k. dow 0=Mon.'),
          open(OUT / 'meta.json', 'w'), indent=1)
if MODE == 'graphs':
    zp = WORK / 'jinay_graphs_outputs.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(str(OUT / '**/*'), recursive=True):
            if os.path.isfile(f): z.write(f, os.path.relpath(f, WORK))
    print('\nzip: %s (%.1f MB)' % (zp, os.path.getsize(zp) / 1e6))
    print('NEXT: Save Version (Commit) > open the committed notebook > Output tab > New Dataset > name it "tcstag-regime-graphs" > share the link with the group.')

# PART B: PEMS08 baseline (only when `MODE = 'baseline'`, GPU session)
Same protocol as Harshith's PEMS04 baseline, so the numbers are comparable. Attach the graph dataset from the `graphs` run (use the **v2** graphs) so the PEMS08 graph is reused and identical to the one Advik trains on; otherwise it is recomputed in Cell 6 (about a minute).

In [ ]:
if MODE == 'baseline':
    GDIR = REPO / 'data' / 'PEMS08'
    prev = find('stag_001_PEMS08_global.csv')
    if prev and Path(prev).parent != GDIR:
        print('reusing existing graphs from', Path(prev).parent)
        shutil.copy(prev, GDIR / 'stag_001_PEMS08.csv'); shutil.copy(find('strg_001_PEMS08_global.csv'), GDIR / 'strg_001_PEMS08.csv')
    assert (GDIR / 'stag_001_PEMS08.csv').exists() and (GDIR / 'strg_001_PEMS08.csv').exists()
    r = subprocess.run([sys.executable, 'prepareData.py', '--config', 'configurations/PEMS08_dstagnn.conf'], cwd=REPO, capture_output=True, text=True)
    print(r.stdout[-600:], r.stderr[-800:]); assert (GDIR / 'PEMS08_r1_d0_w0_dstagnn.npz').exists(), 'prepareData failed'
    RUNNER = r'''# DSTAGNN runner: one training run = one (dataset, graph, seed). Used by every team notebook.
import os, sys, json, time, argparse, random, configparser, shutil

p = argparse.ArgumentParser()
p.add_argument('--repo', default='.')
p.add_argument('--dataset', default='PEMS04')
p.add_argument('--graph', default='AG', help='AG = learned STAG (paper), G = static road graph')
p.add_argument('--seed', type=int, default=1)
p.add_argument('--gpu', default='0')
p.add_argument('--max_epochs', type=int, default=60)
p.add_argument('--patience', type=int, default=15)
p.add_argument('--out', required=True)
p.add_argument('--deadline', type=float, default=0.0, help='unix time; stop training after this')
args = p.parse_args()

os.environ['CUDA_VISIBLE_DEVICES'] = str(args.gpu)
os.chdir(args.repo)
sys.path.insert(0, os.getcwd())
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from model.DSTAGNN_my import make_model
from lib.dataloader import load_weighted_adjacency_matrix, load_PA
from lib.utils1 import load_graphdata_channel1, get_adjacency_matrix2
from lib.metrics import masked_mape_np

random.seed(args.seed); np.random.seed(args.seed)
torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True

cfg = configparser.ConfigParser()
cfg.read('configurations/%s_dstagnn.conf' % args.dataset)
D, T = cfg['Data'], cfg['Training']
N = int(D['num_of_vertices']); n_pred = int(D['num_for_predict']); len_in = int(D['len_input'])
bs = int(T['batch_size']); lr = float(T['learning_rate'])
hours, days, weeks = int(T['num_of_hours']), int(T['num_of_days']), int(T['num_of_weeks'])
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
os.makedirs(args.out, exist_ok=True)

(_, train_loader, _, _, val_loader, _, _, test_loader, test_target, mean, std) = load_graphdata_channel1(
    D['graph_signal_matrix_filename'], hours, days, weeks, DEVICE, bs)
adj_mx = get_adjacency_matrix2(D['adj_filename'], N, id_filename=None)
adj_stag = load_weighted_adjacency_matrix(D['stag_filename'], N)
adj_strg = load_PA(D['strg_filename'])
adj_merge = adj_mx if args.graph == 'G' else adj_stag

net = make_model(DEVICE, int(T['in_channels']), int(T['nb_block']), int(T['in_channels']), int(T['K']),
                 int(T['nb_chev_filter']), int(T['nb_time_filter']), 1, adj_merge, adj_strg, adj_stag,
                 n_pred, len_in, N, int(T['d_model']), int(T['d_k']), int(T['d_k']), int(T['n_heads']))
n_params = int(sum(p_.numel() for p_ in net.parameters()))
print('params:', n_params, 'device:', DEVICE, flush=True)
crit = nn.SmoothL1Loss().to(DEVICE)   # Huber, threshold 1 (as in paper)
opt = optim.Adam(net.parameters(), lr=lr)

def val_loss():
    net.eval(); tot = []
    with torch.no_grad():
        for x, y in val_loader:
            tot.append(crit(net(x), y).item())
    return float(np.mean(tot))

best_path = os.path.join(args.out, 'best.pt')
best_val, best_ep, bad, hist, stopped = float('inf'), -1, 0, [], 'max_epochs'
t_start = time.time()
for ep in range(args.max_epochs):
    t0 = time.time(); net.train(); tl = []
    for x, y in train_loader:
        opt.zero_grad(); loss = crit(net(x), y); loss.backward(); opt.step(); tl.append(loss.detach())
    tr = float(torch.stack(tl).mean()); t_train = time.time() - t0
    vl = val_loss()
    hist.append(dict(epoch=ep, train_loss=tr, val_loss=vl, epoch_sec=time.time() - t0))
    improved = vl < best_val
    if improved:
        best_val, best_ep, bad = vl, ep, 0
        torch.save(net.state_dict(), best_path)
    else:
        bad += 1
    print('ep %d train %.4f val %.4f %s  %.1fs' % (ep, tr, vl, '*' if improved else '', time.time() - t0), flush=True)
    if bad >= args.patience:
        stopped = 'early_stop'; break
    if args.deadline and time.time() > args.deadline:
        stopped = 'deadline'; break

# ---- test at best epoch ----
net.load_state_dict(torch.load(best_path, map_location=DEVICE)); net.eval(); preds = []
with torch.no_grad():
    for x, y in test_loader:
        preds.append(net(x).cpu().numpy())
pred = np.concatenate(preds, 0); tgt = test_target.cpu().numpy()

def m(y, yh):
    return dict(MAE=float(np.mean(np.abs(y - yh))), RMSE=float(np.sqrt(np.mean((y - yh) ** 2))),
                MAPE=float(masked_mape_np(y, yh, 0)))

per_h = [m(tgt[:, :, i], pred[:, :, i]) for i in range(pred.shape[2])]
overall = m(tgt.reshape(-1), pred.reshape(-1))
prep = np.load(D['graph_signal_matrix_filename'].replace('.npz', '') + '_r%d_d%d_w%d_dstagnn.npz' % (hours, days, weeks))
np.savez_compressed(os.path.join(args.out, 'test_preds.npz'), prediction=pred.astype(np.float32),
                    target=tgt.astype(np.float32), timestamp=prep['test_timestamp'])
res = dict(dataset=args.dataset, graph=args.graph, seed=args.seed, best_epoch=best_ep, best_val=best_val,
           epochs_run=len(hist), stopped_by=stopped, max_epochs=args.max_epochs, patience=args.patience,
           mean_epoch_sec=float(np.mean([h['epoch_sec'] for h in hist])), total_sec=time.time() - t_start,
           n_params=n_params, test_overall=overall, test_per_horizon=per_h, history=hist)
json.dump(res, open(os.path.join(args.out, 'results.json'), 'w'), indent=1)
os.remove(best_path)   # keep outputs small; predictions + results are what we need
print('TEST overall:', overall, flush=True)
'''
    (WORK / 'runner.py').write_text(RUNNER); print('runner written')

In [ ]:
DATASET, GRAPH = 'PEMS08', 'AG'
def launch(seed, gpu, max_epochs, patience, out, graph=GRAPH):
    cmd = [sys.executable, str(WORK / 'runner.py'), '--repo', str(REPO), '--dataset', DATASET, '--graph', graph, '--seed', str(seed),
           '--gpu', str(gpu), '--max_epochs', str(max_epochs), '--patience', str(patience), '--out', str(out), '--deadline', str(SESSION_DEADLINE)]
    return subprocess.run(cmd, stdout=open(str(out) + '.log', 'w'), stderr=subprocess.STDOUT)

if MODE == 'baseline' and RUN_TIMING_TEST:
    out = RUNS / 'timing_test'; out.mkdir(exist_ok=True, parents=True); p = launch(0, 0, 1, 1, out)
    print(open(str(out) + '.log').read()[-1200:]); assert p.returncode == 0, 'Timing run crashed: send the traceback above to the group.'
    r = json.load(open(out / 'results.json')); sec = r['mean_epoch_sec']
    print('\
>>> TIMING (PEMS08): %.0f s/epoch | %d params' % (sec, r['n_params']))
    print('>>> up to %.1f h per seed; %d seeds on %d GPU(s) = about %.1f h wall clock' % (sec * MAX_EPOCHS / 3600, len(SEEDS), N_GPU, sec * MAX_EPOCHS / 3600 * -(-len(SEEDS) // N_GPU)))

### Train PEMS08 baseline, seeds 1-3, one seed per GPU (finished seeds are skipped on rerun)

In [ ]:
if MODE == 'baseline':
    jobs = queue.Queue(); [jobs.put(s) for s in SEEDS]; status = {}
    def worker(gpu):
        while True:
            try: s = jobs.get_nowait()
            except queue.Empty: return
            out = RUNS / ('%s_%s_seed%d' % (DATASET, GRAPH, s)); out.mkdir(exist_ok=True, parents=True)
            if (out / 'results.json').exists(): status[s] = 'skipped'; continue
            t0 = time.time(); p = launch(s, gpu, MAX_EPOCHS, PATIENCE, out)
            status[s] = 'ok' if p.returncode == 0 else 'FAILED (see %s.log)' % out
            print('seed %d on GPU %d -> %s in %.1f min' % (s, gpu, status[s], (time.time() - t0) / 60), flush=True)
    th = [threading.Thread(target=worker, args=(g,)) for g in range(min(N_GPU, len(SEEDS)))]
    [t.start() for t in th]; [t.join() for t in th]; print(status)

### Results vs paper, per-horizon error, and error by regime (PEMS08)

In [ ]:
if MODE == 'baseline':
    rows, hrows = [], []
    for f in sorted(glob.glob(str(RUNS / ('%s_%s_seed*/results.json' % (DATASET, GRAPH))))):
        r = json.load(open(f)); o = r['test_overall']
        rows.append(dict(seed=r['seed'], MAE=o['MAE'], MAPE=o['MAPE'], RMSE=o['RMSE'], best_epoch=r['best_epoch'], epochs_run=r['epochs_run'], stopped_by=r['stopped_by'], sec_per_epoch=r['mean_epoch_sec']))
        hrows += [dict(seed=r['seed'], horizon=i + 1, **h) for i, h in enumerate(r['test_per_horizon'])]
    res = pd.DataFrame(rows); print(res.round(3).to_string(index=False))
    summ = pd.DataFrame({'ours_mean': res[['MAE', 'MAPE', 'RMSE']].mean(), 'ours_std': res[['MAE', 'MAPE', 'RMSE']].std(), 'paper': pd.Series(PAPER['PEMS08'])})
    summ['gap_vs_paper_%'] = 100 * (summ.ours_mean - summ.paper) / summ.paper; print('\n', summ.round(3))
    res.to_csv(WORK / 'baseline_summary_PEMS08.csv', index=False); pd.DataFrame(hrows).to_csv(WORK / 'baseline_per_horizon_PEMS08.csv', index=False)
    if (summ['gap_vs_paper_%'].abs() > 5).any(): print('\nNOTE: >5% off the paper. Likely causes: our own PEMS08 graph (the authors do not ship theirs) or the epoch cap. Tell the group.')

    def regime_of(t, ds='PEMS08'):
        tod = t % 288; dow = ((t // 288) + START_DOW[ds]) % 7
        peak = ((tod >= 84) & (tod < 120)) | ((tod >= 192) & (tod < 228))
        return np.where(dow >= 5, 'weekend', np.where(peak, 'weekday_peak', 'weekday_offpeak'))
    recs = []
    for f in sorted(glob.glob(str(RUNS / ('%s_%s_seed*/test_preds.npz' % (DATASET, GRAPH))))):
        z = np.load(f); reg = regime_of(z['timestamp'][:, 0].astype(int)); seed = int(f.split('seed')[1].split('/')[0])
        for name in REG:
            m = reg == name; e = z['prediction'][m] - z['target'][m]
            recs.append(dict(seed=seed, regime=name, n_windows=int(m.sum()), MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean())))
    rg = pd.DataFrame(recs); print(rg.groupby('regime')[['n_windows', 'MAE', 'RMSE']].mean().round(2)); rg.to_csv(WORK / 'baseline_by_regime_PEMS08.csv', index=False)

    for f in glob.glob(str(RUNS / '*/best.pt')): os.remove(f)
    shutil.rmtree(RUNS / 'timing_test', ignore_errors=True)
    zp = WORK / 'jinay_pems08_baseline_outputs.zip'
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in glob.glob(str(RUNS / '*/*')) + glob.glob(str(RUNS / '*.log')) + glob.glob(str(WORK / 'baseline_*.csv')): z.write(f, os.path.relpath(f, WORK))
    print('zip:', zp, '| %.1f MB' % (os.path.getsize(zp) / 1e6))
    print('Send to the group: the zip, the TIMING line, the results table, the regime table.')
else:
    print('MODE is graphs: nothing to do here. Switch MODE to baseline on a GPU session for Part B.')